# Índices de mercado 2026: quem venceu a inflação?

Análise de retorno e risco de 9 índices que representam as principais classes de ativos de um investidor brasileiro. A lógica está em `src/indices/`; aqui cada etapa é executada e explicada.

**Perguntas:** 1) quanto cada classe rendeu no ano e em 12 meses, em termos nominais e reais? 2) qual foi o risco para chegar lá (volatilidade, maior queda)? 3) o retorno compensou o risco frente ao CDI?

In [1]:
import sys; sys.path.insert(0, "../src")
import pandas as pd
pd.set_option("display.precision", 2)
from indices import coleta, metricas

## 1. Dados
A base mensal vem de relatório de mercado; CDI, IPCA e dólar são baixados da API pública do Banco Central (SGS) e substituem a base quando a API responde. O CDI chega como taxa **diária** e é encadeado para o mês; o dólar vira variação do último valor de cada mês.

In [2]:
longo = coleta.montar()
largo = coleta.em_colunas(longo)
print("fontes:", sorted(longo.fonte.unique()), "| preliminares:", longo[longo.previa][["indice", "mes"]].values.tolist())
largo

API do BCB indisponível (ProxyError); usando só a base.
fontes: ['base'] | preliminares: [['IPCA', Period('2026-10', 'M')]]


indice,IBOV,S&P 500,IMA-B,CDI,IFIX,IRF-M,IPCA,IHFA,Dólar
mes,,,,,,,,,
2025-10,NaN,NaN,NaN,NaN,NaN,NaN,0.09,NaN,NaN
2025-11,6.37,0.13,2.04,1.05,1.86,1.67,0.18,1.38,-0.94
2025-12,1.29,-0.05,0.31,1.22,3.14,0.30,0.33,0.54,3.16
2026-01,12.56,1.37,1.00,1.16,2.27,1.96,0.33,2.23,-4.95
2026-02,4.09,-0.87,1.79,1.00,1.32,0.99,0.70,1.33,-1.54
2026-03,-0.70,-5.09,0.17,1.21,-1.06,-0.59,0.88,-3.42,1.36
2026-04,-0.08,10.42,1.81,1.09,1.53,1.24,0.67,2.05,-4.42
2026-05,-7.22,5.15,0.31,1.07,-1.33,0.68,0.58,0.62,1.37
2026-06,-1.01,-1.06,-1.04,1.12,-1.21,0.69,0.16,0.51,2.37


## 2. Retorno e risco
**Retornos se compõem, não se somam:** 1% + 1% em dois meses dá 2,01%, não 2%. Retorno real = (1 + nominal) / (1 + IPCA) − 1. Volatilidade anualizada = desvio-padrão mensal × √12. Sharpe = excesso médio sobre o CDI ÷ desvio do excesso × √12.

In [3]:
res = metricas.resumo(largo, 2026)
res

,indice,no_ano_2026,real_no_ano,ult_12m,real_12m,vol_anual,sharpe_12m,max_drawdown_12m,meses_positivos
0,CDI,11.68,7.09,14.23,8.98,0.25,NaN,0.00,12
1,S&P 500,10.74,6.19,10.83,5.74,13.31,-0.17,-5.92,5
2,IMA-B,8.98,4.50,11.54,6.42,4.33,-0.53,-1.04,10
3,IRF-M,7.63,3.21,9.75,4.71,3.56,-1.10,-1.52,10
4,IHFA,5.81,1.47,7.85,2.90,5.65,-0.98,-3.42,9
5,IPCA,4.28,NaN,4.82,NaN,1.21,NaN,-0.32,11
6,IBOV,4.22,-0.06,12.29,7.13,20.63,0.01,-11.05,6
7,Dólar,-1.47,-5.52,0.69,-3.94,10.23,-1.19,-9.33,6
8,IFIX,-3.41,-7.38,1.47,-3.19,6.38,-1.85,-7.21,5


![no ano](../outputs/no_ano.png)

## 3. Evolução no ano

![evolução](../outputs/evolucao_ano.png)

## 4. Mês a mês

![mapa](../outputs/mapa_mensal.png)

## 5. Meta IPCA + 4% ao ano
Meta comum em planejamento de longo prazo. A taxa anual vira mensal por (1 + 4%)^(1/12) − 1 e é encadeada com o IPCA de cada mês.

In [4]:
ano = largo[largo.index.year == 2026]
meta = metricas.acumulado(metricas.benchmark_ipca_mais(ano, 4.0))
print(f"Meta IPCA + 4% a.a. em 2026: {meta:.2f}%")
res.assign(bateu_meta=res["no_ano_2026"] > meta)[["indice", "no_ano_2026", "bateu_meta"]]

Meta IPCA + 4% a.a. em 2026: 7.75%


,indice,no_ano_2026,bateu_meta
0,CDI,11.68,True
1,S&P 500,10.74,True
2,IMA-B,8.98,True
3,IRF-M,7.63,False
4,IHFA,5.81,False
5,IPCA,4.28,False
6,IBOV,4.22,False
7,Dólar,-1.47,False
8,IFIX,-3.41,False


## 6. Correlação entre as classes (12 meses)
Diversificar funciona quando os ativos não sobem e caem juntos.

In [5]:
largo.tail(12).corr().round(2)

indice,IBOV,S&P 500,IMA-B,CDI,IFIX,IRF-M,IPCA,IHFA,Dólar
indice,,,,,,,,,
IBOV,1.00,-0.10,0.59,0.15,0.72,0.77,-0.02,0.49,-0.71
S&P 500,-0.10,1.00,0.27,-0.28,0.20,0.35,-0.05,0.59,-0.43
IMA-B,0.59,0.27,1.00,-0.31,0.44,0.77,0.19,0.61,-0.57
CDI,0.15,-0.28,-0.31,1.00,0.15,-0.19,-0.10,-0.44,0.03
IFIX,0.72,0.20,0.44,0.15,1.00,0.55,0.12,0.48,-0.57
IRF-M,0.77,0.35,0.77,-0.19,0.55,1.00,-0.05,0.83,-0.69
IPCA,-0.02,-0.05,0.19,-0.10,0.12,-0.05,1.00,-0.20,-0.19
IHFA,0.49,0.59,0.61,-0.44,0.48,0.83,-0.20,1.00,-0.50
Dólar,-0.71,-0.43,-0.57,0.03,-0.57,-0.69,-0.19,-0.50,1.00


## 7. Leitura e limitações
- Com 12 meses, volatilidade, Sharpe e correlação são **estimativas instáveis**: um único mês ruim muda bastante os números. Servem para comparar classes, não para prever.
- A base não informa se o S&P 500 está em dólar ou em reais; a comparação com índices brasileiros deve considerar isso.
- O IPCA do último mês pode ser a prévia (IPCA-15) até o IBGE divulgar o índice cheio; o pipeline marca esses valores.
- Análise descritiva, não recomendação de investimento.